In [5]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface \
               faiss-cpu tiktoken python-dotenv sentence-transformers langchain-groq

In [9]:
!pip install -q langchain langchain-text-splitters

In [15]:
!pip install -q "youtube-transcript-api==0.6.2"

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from google.colab import userdata


In [18]:

# ---------------- 0) Get video ID from user (URL or raw ID) ----------------
def extract_video_id(url_or_id: str) -> str:
    patterns = [r"(?:v=|youtu\.be/|embed/)([A-Za-z0-9_-]{11})"]
    for pattern in patterns:
        match = re.search(pattern, url_or_id)
        if match:
            return match.group(1)
    return url_or_id.strip()

user_input_raw = input("Enter the YouTube video URL or ID: ").strip()
video_id = extract_video_id(user_input_raw)
print("Using video ID:", video_id)

# ---------------- 1) Get transcript ----------------
transcript = None
try:
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript = " ".join(snippet.text for snippet in fetched_transcript)
    print(transcript[:500], "...")
except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"Could not fetch transcript: {e}")

if not transcript:
    raise SystemExit("Stopping: no transcript to process.")

# ---------------- 2) Split into chunks ----------------
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print("Number of chunks:", len(chunks))

# ---------------- 3) Create embeddings using sentence-transformers ----------------
# HuggingFaceEmbeddings wraps sentence-transformers in a format LangChain/FAISS understands
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build the vector store from the chunks + embedding model
vector_store = FAISS.from_documents(chunks, embedding_model)
print("Vector store built. Total vectors:", len(vector_store.index_to_docstore_id))

# ---------------- 4) Set up retriever ----------------
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# ---------------- 5) LLM (Groq) ----------------


Enter the YouTube video URL or ID: https://www.youtube.com/watch?v=ktNeWrkPwmg
Using video ID: ktNeWrkPwmg
hey wouldn't it be nice to have a tool to turn any audio to text with great accuracy and completely free and you don't need to download anything to your local computer well today I'm going to show you exactly how you can do that using open ai's whisper AI right in your browser with Google col hi if you're new here my name is l whether you wanted to transcribe podcasts interviews or even YouTube videos whisper AI makes it effortless oh and you don't need to know how to code I'm also going to shar ...
Number of chunks: 7


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store built. Total vectors: 7
Build the parallel chain and ask a question? (yes/no): yes
Ask something about the video: what is this video about

Answer:
 This video is about using Open AI's Whisper AI in Google Colab to transcribe audio or video files to text with great accuracy and for free, without needing to download anything to your local computer.


In [19]:
groq_api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.3-70b-versatile",
               api_key=groq_api_key)

# ---------------- 6) Prompt ----------------
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

Context:
{context}

Question: {question}
""",
    input_variables=["context", "question"]
)

# ---------------- 7) Build the parallel chain ----------------
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

# This is where the embedding model comes in indirectly:
# retriever uses embedding_model internally to find similar chunks to the question
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

parser = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser

# ---------------- 8) Ask user + get output ----------------
build_chain = input("Build the parallel chain and ask a question? (yes/no): ").strip().lower()

if build_chain in ("yes", "y"):
    user_question = input("Ask something about the video: ")
    result = main_chain.invoke(user_question)
    print("\nAnswer:\n", result)
else:
    print("Skipped. Vector store and embeddings are ready if you want to query later.")

Build the parallel chain and ask a question? (yes/no): yes
Ask something about the video: what ai tool she use

Answer:
 She used Open AI's Whisper AI.
